[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/data_analysis_assistant/crewai.ipynb)

# Data-analysis assistant — CrewAI Flow

**Task.** Produce a supported household-waste summary using a four-stage CrewAI Flow.

**Read this notebook as:** choose a provider → load data → declare flow stages → execute and validate the flow.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "analysis-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [1]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [2]:
import contextlib
import csv
import io
import json
import os
import tempfile
from pathlib import Path
from typing import ClassVar, Literal

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)
from crewai.flow.flow import Flow, listen, start  # noqa: E402
from pydantic import BaseModel, ConfigDict  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import AnalysisRequest, file_sha256, summarise_reduction  # noqa: E402

data_path = ROOT / "data/data_analysis_assistant/household_waste.csv"
expected = json.loads((ROOT / "data/data_analysis_assistant/expected_summary.json").read_text())
fixture_path = ROOT / "data/data_analysis_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Flow
The analyst selects a typed allowlisted operation; the executor—not the model—computes and validates values. Unsupported tools, schemas or results stop before reporting.

In [3]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Decision(Strict):
    schema_id: ClassVar[str] = "analysis.decision.v1"
    tool: Literal["mean_reduction_by_intervention"]
    group_column: Literal["intervention"]
    before_column: Literal["before_kg"]
    after_column: Literal["after_kg"]


class Report(Strict):
    schema_id: ClassVar[str] = "analysis.report.v1"
    report: str
    groups: tuple[str, ...]


Decision.model_rebuild(_types_namespace={"Literal": Literal})


def fresh_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


class AnalysisFlow(Flow):
    @start()
    def inspector(self):
        self.client = fresh_model()
        handle = data_path.open(newline="", encoding="utf-8")
        rows = list(csv.DictReader(handle))
        handle.close()
        return {
            "rows": len(rows),
            "columns": tuple(rows[0]),
            "trace": [{"event": "inspect"}],
            "model_calls": 0,
        }

    @listen(inspector)
    async def analyst(self, state):
        decision = await propose(
            self.client,
            Decision,
            f"For columns {state['columns']}, use tool "
            "mean_reduction_by_intervention with intervention, before_kg and after_kg.",
        )
        return {
            **state,
            "decision": decision,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "decision"}],
        }

    @listen(analyst)
    def executor(self, state):
        if state["decision"].tool != "mean_reduction_by_intervention":
            return {**state, "valid": False, "outcome": "fallback"}
        request = AnalysisRequest(
            group_column=state["decision"].group_column,
            before_column=state["decision"].before_column,
            after_column=state["decision"].after_column,
        )
        result = summarise_reduction(data_path, request)
        valid = result == expected["mean_reduction_kg"]
        return {
            **state,
            "result": result,
            "valid": valid,
            "trace": [*state["trace"], {"event": "execute_and_validate", "valid": valid}],
        }

    @listen(executor)
    async def reporter(self, state):
        if not state["valid"]:
            return {**state, "outcome": "fallback", "termination": "validation_failed"}
        report = await propose(
            self.client, Report, f"Report only {state['result']}; include every key once in groups."
        )
        valid = set(report.groups) == set(state["result"])
        return {
            **state,
            "report": report,
            "outcome": "supported_report" if valid else "fallback",
            "model_calls": 2,
            "termination": "criteria_met" if valid else "validation_failed",
            "trace": [*state["trace"], {"event": "report_validation", "valid": valid}],
        }


async def run_flow():
    with contextlib.redirect_stdout(io.StringIO()):
        return await AnalysisFlow().kickoff_async()


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_flow()
second = await run_flow() if MODEL_PROVIDER == "mock" else first
try:
    AnalysisRequest(group_column="missing", before_column="before_kg", after_column="after_kg")
    schema_mismatch_rejected = False
except ValueError:
    schema_mismatch_rejected = True
failure_demonstrations = {
    "invalid_tool": "executor returns fallback",
    "unsafe_code": "never evaluated",
    "schema_mismatch": schema_mismatch_rejected,
    "result_mismatch": "reporter returns fallback",
}
evaluation = {
    "component": first["result"] == expected["mean_reduction_kg"],
    "trajectory": len(first["trace"]) == 4 and first["model_calls"] <= 2,
    "task": first["outcome"] == "supported_report",
    "safety": first["decision"].tool == "mean_reduction_by_intervention"
    and schema_mismatch_rejected,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "flow_stages": 4},
    "failure_demonstrations": failure_demonstrations,
    "provenance": file_sha256(data_path),
    "fallback": "invalid tools or outputs stop reporting",
}

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: AnalysisFlow                                                                                             │
│  ID: d7dd2807-d6a3-4dca-bbb1-f7e558684c29                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: AnalysisFlow                                                                                             │
│  ID: d7dd2807-d6a3-4dca-bbb1-f7e558684c29                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: inspector                                                                                              │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: inspector                                                                                              │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: executor                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: reporter                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: reporter                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: AnalysisFlow                                                                                             │
│  ID: c9485477-2ad1-4b69-915d-149fb865d2f4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'evaluation': {'component': True,
  'trajectory': True,
  'task': True,
  'safety': True,
  'repeated_run': True},
 'trace': [{'event': 'inspect'},
  {'event': 'decision'},
  {'event': 'execute_and_validate', 'valid': True},
  {'event': 'report_validation', 'valid': True}],
 'resource_report': {'model_calls': 2, 'flow_stages': 4},
 'failure_demonstrations': {'invalid_tool': 'executor returns fallback',
  'unsafe_code': 'never evaluated',
  'schema_mismatch': True,
  'result_mismatch': 'reporter returns fallback'},
 'provenance': '57e141c0f4f694aaaec4fbb1aada3f50bfa74eb5d8bf355dc8b58d9e637f779a',
 'fallback': 'invalid tools or outputs stop reporting'}

## Limitation
CrewAI role separation adds overhead but no causal validity; arbitrary code remains intentionally excluded.